# 05 — Improved Text Teacher Experiment

This notebook runs the next experiment after the first Breakfast proof-of-concept.

Goal:

1. strengthen the visual-only CE student;
2. train a simpler text-aware teacher using concatenated video + text context;
3. distill this teacher into a video-only student;
4. save predictions;
5. compute TAS metrics: Accuracy, Edit, F1@10, F1@25, F1@50.

Main comparison:

```text
student_ce_only_strong
vs
student_kd_from_concat_teacher
```

The final KD student remains **video-only at inference**.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Imports, seed, and paths

In [ ]:
from pathlib import Path
import json
import random
import shutil
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

DRIVE_ROOT = Path('/content/drive/MyDrive/mmf_tas_lab_data')

BREAKFAST_ROOT = DRIVE_ROOT / 'zenodo_ms_tcn_data' / 'breakfast'
FEATURE_DIR = BREAKFAST_ROOT / 'features'
GT_DIR = BREAKFAST_ROOT / 'groundTruth'
SPLIT_DIR = BREAKFAST_ROOT / 'splits'
MAPPING_PATH = BREAKFAST_ROOT / 'mapping.txt'

TEXT_ROOT = DRIVE_ROOT / 'text_assisted_tas' / 'breakfast' / 'text_embeddings'
TEXT_EMB_PATH = TEXT_ROOT / 'breakfast_clip_vitb16_text_embeddings.npy'
TEXT_META_PATH = TEXT_ROOT / 'breakfast_clip_vitb16_text_embedding_metadata.csv'

RUN_BASE = DRIVE_ROOT / 'text_assisted_tas' / 'breakfast' / 'runs'

assert BREAKFAST_ROOT.exists(), BREAKFAST_ROOT
assert FEATURE_DIR.exists(), FEATURE_DIR
assert GT_DIR.exists(), GT_DIR
assert SPLIT_DIR.exists(), SPLIT_DIR
assert MAPPING_PATH.exists(), MAPPING_PATH
assert TEXT_EMB_PATH.exists(), TEXT_EMB_PATH

print('BREAKFAST_ROOT:', BREAKFAST_ROOT)
print('TEXT_ROOT:', TEXT_ROOT)
print('RUN_BASE:', RUN_BASE)

device: cuda
BREAKFAST_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast
TEXT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/text_embeddings
RUN_BASE: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs


## 3. Experiment configuration

Start with `scale1_control`. If it runs correctly, change to `full_split1`.

In [ ]:
# RUN_MODE = 'scale1_control'
# RUN_MODE = 'local_debug'
RUN_MODE = 'full_split1'

SPLIT_ID = 1

CONFIGS = {
    'local_debug': {
        'max_train_videos': 20,
        'max_test_videos': 10,
        'epochs_ce': 1,
        'epochs_teacher': 1,
        'epochs_kd': 1,
        'num_stages': 2,
        'num_layers': 6,
        'num_f_maps': 64,
    },
    'scale1_control': {
        'max_train_videos': 200,
        'max_test_videos': 50,
        'epochs_ce': 5,
        'epochs_teacher': 5,
        'epochs_kd': 5,
        'num_stages': 3,
        'num_layers': 10,
        'num_f_maps': 64,
    },
    'full_split1': {
        'max_train_videos': None,
        'max_test_videos': None,
        'epochs_ce': 10,
        'epochs_teacher': 10,
        'epochs_kd': 10,
        'num_stages': 3,
        'num_layers': 10,
        'num_f_maps': 64,
    },
}

cfg = CONFIGS[RUN_MODE]

MAX_TRAIN_VIDEOS = cfg['max_train_videos']
MAX_TEST_VIDEOS = cfg['max_test_videos']

EPOCHS_CE = cfg['epochs_ce']
EPOCHS_TEACHER = cfg['epochs_teacher']
EPOCHS_KD = cfg['epochs_kd']

NUM_STAGES = cfg['num_stages']
NUM_LAYERS = cfg['num_layers']
NUM_F_MAPS = cfg['num_f_maps']

BATCH_SIZE = 1
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5

KD_TEMPERATURE = 4.0
LAMBDA_CE = 1.0
LAMBDA_KD = 0.1

RUN_NAME = f'improved_concat_teacher_{RUN_MODE}_split{SPLIT_ID}'
RUN_ROOT = RUN_BASE / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'RUN_MODE': RUN_MODE,
    'RUN_NAME': RUN_NAME,
    'SPLIT_ID': SPLIT_ID,
    'MAX_TRAIN_VIDEOS': MAX_TRAIN_VIDEOS,
    'MAX_TEST_VIDEOS': MAX_TEST_VIDEOS,
    'EPOCHS_CE': EPOCHS_CE,
    'EPOCHS_TEACHER': EPOCHS_TEACHER,
    'EPOCHS_KD': EPOCHS_KD,
    'NUM_STAGES': NUM_STAGES,
    'NUM_LAYERS': NUM_LAYERS,
    'NUM_F_MAPS': NUM_F_MAPS,
    'LAMBDA_CE': LAMBDA_CE,
    'LAMBDA_KD': LAMBDA_KD,
    'KD_TEMPERATURE': KD_TEMPERATURE,
    'RUN_ROOT': str(RUN_ROOT),
}, indent=2))

{
  "RUN_MODE": "full_split1",
  "RUN_NAME": "improved_concat_teacher_full_split1_split1",
  "SPLIT_ID": 1,
  "MAX_TRAIN_VIDEOS": null,
  "MAX_TEST_VIDEOS": null,
  "EPOCHS_CE": 10,
  "EPOCHS_TEACHER": 10,
  "EPOCHS_KD": 10,
  "NUM_STAGES": 3,
  "NUM_LAYERS": 10,
  "NUM_F_MAPS": 64,
  "LAMBDA_CE": 1.0,
  "LAMBDA_KD": 0.1,
  "KD_TEMPERATURE": 4.0,
  "RUN_ROOT": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1"
}


## 4. Load mapping, text embeddings, and splits

In [ ]:
def load_mapping(mapping_path):
    id_to_label = {}
    label_to_id = {}

    for line in Path(mapping_path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        class_id = int(parts[0])
        label = parts[1]
        id_to_label[class_id] = label
        label_to_id[label] = class_id

    return id_to_label, label_to_id


def find_split_file(split_dir, split_id, train=True):
    prefix = 'train' if train else 'test'
    candidates = [
        split_dir / f'{prefix}.split{split_id}.bundle',
        split_dir / f'{prefix}.split{split_id}.txt',
        split_dir / f'{prefix}{split_id}.bundle',
        split_dir / f'{prefix}{split_id}.txt',
    ]

    for p in candidates:
        if p.exists():
            return p

    matches = sorted(split_dir.glob(f'*{prefix}*{split_id}*'))
    if matches:
        return matches[0]

    raise FileNotFoundError(f'Could not find {prefix} split {split_id} in {split_dir}')


def read_split_ids(path):
    ids = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        video_id = Path(line).stem
        ids.append(video_id)
    return ids


id_to_label, label_to_id = load_mapping(MAPPING_PATH)
num_classes = len(id_to_label)
sil_id = label_to_id.get('SIL', None)

text_embeddings = np.load(TEXT_EMB_PATH).astype(np.float32)
text_metadata = pd.read_csv(TEXT_META_PATH) if TEXT_META_PATH.exists() else None

print('num_classes:', num_classes)
print('sil_id:', sil_id)
print('text_embeddings:', text_embeddings.shape)

train_split_path = find_split_file(SPLIT_DIR, SPLIT_ID, train=True)
test_split_path = find_split_file(SPLIT_DIR, SPLIT_ID, train=False)

train_ids = read_split_ids(train_split_path)
test_ids = read_split_ids(test_split_path)

if MAX_TRAIN_VIDEOS is not None:
    train_ids = train_ids[:MAX_TRAIN_VIDEOS]
if MAX_TEST_VIDEOS is not None:
    test_ids = test_ids[:MAX_TEST_VIDEOS]

print('train_split_path:', train_split_path)
print('test_split_path:', test_split_path)
print('train videos:', len(train_ids))
print('test videos:', len(test_ids))
print('first train:', train_ids[:3])
print('first test:', test_ids[:3])

num_classes: 48
sil_id: 0
text_embeddings: (48, 512)
train_split_path: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/splits/train.split1.bundle
test_split_path: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast/splits/test.split1.bundle
train videos: 1460
test videos: 252
first train: ['P16_cam01_P16_cereals', 'P16_cam01_P16_friedegg', 'P16_cam01_P16_juice']
first test: ['P03_cam01_P03_cereals', 'P03_cam01_P03_coffee', 'P03_cam01_P03_friedegg']


## 5. Dataset and DataLoader

For the teacher, each sample includes a `text_context` vector.

Here `text_context` is the mean CLIP embedding of action classes present in the video. This is privileged train/eval text for the teacher. The KD student does not receive this vector at inference.

In [ ]:
class BreakfastTASDataset(Dataset):
    def __init__(self, video_ids, feature_dir, gt_dir, label_to_id, id_to_label, text_embeddings, sil_id=None):
        self.video_ids = list(video_ids)
        self.feature_dir = Path(feature_dir)
        self.gt_dir = Path(gt_dir)
        self.label_to_id = label_to_id
        self.id_to_label = id_to_label
        self.text_embeddings = text_embeddings
        self.sil_id = sil_id
        self.text_dim = int(text_embeddings.shape[1])

    def __len__(self):
        return len(self.video_ids)

    def _load_features(self, video_id):
        path = self.feature_dir / f'{video_id}.npy'
        x = np.load(path).astype(np.float32)

        if x.ndim != 2:
            raise ValueError(f'Expected 2D feature array for {video_id}, got {x.shape}')

        if x.shape[0] == 2048:
            pass
        elif x.shape[1] == 2048:
            x = x.T

        return x

    def _load_labels(self, video_id):
        path = self.gt_dir / f'{video_id}.txt'
        labels = []
        for line in path.read_text().splitlines():
            label = line.strip()
            if not label:
                continue
            if label not in self.label_to_id:
                raise KeyError(f'Unknown label in {video_id}: {label}')
            labels.append(self.label_to_id[label])
        return np.asarray(labels, dtype=np.int64)

    def _make_text_context(self, label_ids):
        unique_ids = sorted(set(int(x) for x in label_ids.tolist()))

        if self.sil_id is not None:
            unique_ids = [x for x in unique_ids if x != self.sil_id]

        valid_ids = [x for x in unique_ids if 0 <= x < len(self.text_embeddings)]

        if len(valid_ids) == 0:
            return np.zeros((self.text_dim,), dtype=np.float32)

        context = self.text_embeddings[valid_ids].mean(axis=0)
        return context.astype(np.float32)

    def __getitem__(self, idx):
        video_id = self.video_ids[idx]

        features = self._load_features(video_id)
        labels = self._load_labels(video_id)

        T = min(features.shape[1], len(labels))
        features = features[:, :T]
        labels = labels[:T]

        text_context = self._make_text_context(labels)

        return {
            'video_id': video_id,
            'features': torch.from_numpy(features),
            'labels': torch.from_numpy(labels),
            'text_context': torch.from_numpy(text_context),
            'length': T,
        }


def collate_tas_batch(batch):
    batch_size = len(batch)
    feat_dim = batch[0]['features'].shape[0]
    text_dim = batch[0]['text_context'].shape[0]
    max_T = max(item['length'] for item in batch)

    features = torch.zeros(batch_size, feat_dim, max_T, dtype=torch.float32)
    labels = torch.full((batch_size, max_T), fill_value=-100, dtype=torch.long)
    mask = torch.zeros(batch_size, max_T, dtype=torch.float32)
    text_context = torch.zeros(batch_size, text_dim, dtype=torch.float32)
    lengths = []
    video_ids = []

    for i, item in enumerate(batch):
        T = item['length']
        features[i, :, :T] = item['features']
        labels[i, :T] = item['labels']
        mask[i, :T] = 1.0
        text_context[i] = item['text_context']
        lengths.append(T)
        video_ids.append(item['video_id'])

    return {
        'features': features,
        'labels': labels,
        'mask': mask,
        'text_context': text_context,
        'lengths': torch.tensor(lengths, dtype=torch.long),
        'video_ids': video_ids,
    }


train_dataset = BreakfastTASDataset(
    train_ids, FEATURE_DIR, GT_DIR, label_to_id, id_to_label, text_embeddings, sil_id=sil_id
)
test_dataset = BreakfastTASDataset(
    test_ids, FEATURE_DIR, GT_DIR, label_to_id, id_to_label, text_embeddings, sil_id=sil_id
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_tas_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_tas_batch,
)

sample = train_dataset[0]
feature_dim = sample['features'].shape[0]
text_dim = sample['text_context'].shape[0]

print('feature_dim:', feature_dim)
print('text_dim:', text_dim)
print('sample video:', sample['video_id'])
print('sample features:', sample['features'].shape)
print('sample labels:', sample['labels'].shape)
print('sample text_context:', sample['text_context'].shape)

feature_dim: 2048
text_dim: 512
sample video: P16_cam01_P16_cereals
sample features: torch.Size([2048, 544])
sample labels: torch.Size([544])
sample text_context: torch.Size([512])


## 6. Model definitions

In [ ]:
class DilatedResidualLayer(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.conv_dilated = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.conv_1x1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, mask):
        out = F.relu(self.conv_dilated(x))
        out = self.conv_1x1(out)
        out = self.dropout(out)
        out = (x + out) * mask.unsqueeze(1)
        return out


class SingleStageTCN(nn.Module):
    def __init__(self, in_dim, num_f_maps, num_classes, num_layers):
        super().__init__()
        self.conv_in = nn.Conv1d(in_dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([
            DilatedResidualLayer(num_f_maps, dilation=2 ** i)
            for i in range(num_layers)
        ])
        self.conv_out = nn.Conv1d(num_f_maps, num_classes, kernel_size=1)

    def forward(self, x, mask):
        out = self.conv_in(x)
        out = out * mask.unsqueeze(1)

        for layer in self.layers:
            out = layer(out, mask)

        logits = self.conv_out(out) * mask.unsqueeze(1)
        return logits


class MultiStageTCN(nn.Module):
    def __init__(self, in_dim, num_f_maps, num_classes, num_layers, num_stages):
        super().__init__()
        self.stage1 = SingleStageTCN(in_dim, num_f_maps, num_classes, num_layers)
        self.stages = nn.ModuleList([
            SingleStageTCN(num_classes, num_f_maps, num_classes, num_layers)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x, mask):
        outputs = []

        out = self.stage1(x, mask)
        outputs.append(out)

        for stage in self.stages:
            out = stage(F.softmax(out, dim=1) * mask.unsqueeze(1), mask)
            outputs.append(out)

        return outputs


class ConcatTextTeacher(nn.Module):
    def __init__(self, visual_dim, text_dim, num_f_maps, num_classes, num_layers, num_stages):
        super().__init__()

        self.text_projection = nn.Sequential(
            nn.Linear(text_dim, text_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.tcn = MultiStageTCN(
            in_dim=visual_dim + text_dim,
            num_f_maps=num_f_maps,
            num_classes=num_classes,
            num_layers=num_layers,
            num_stages=num_stages,
        )

    def forward(self, features, mask, text_context):
        B, _, T = features.shape

        text_proj = self.text_projection(text_context)
        text_seq = text_proj.unsqueeze(-1).expand(B, -1, T)

        x = torch.cat([features, text_seq], dim=1)
        return self.tcn(x, mask)

## 7. Losses, training, and evaluation

In [ ]:
ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

def supervised_loss(outputs, labels):
    loss = 0.0
    for logits in outputs:
        loss = loss + ce_loss_fn(logits.permute(0, 2, 1).reshape(-1, logits.shape[1]), labels.reshape(-1))
    return loss / len(outputs)


def kd_loss(student_logits, teacher_logits, mask, temperature=4.0):
    s = student_logits / temperature
    t = teacher_logits / temperature

    log_p_s = F.log_softmax(s, dim=1).permute(0, 2, 1).reshape(-1, s.shape[1])
    p_t = F.softmax(t, dim=1).permute(0, 2, 1).reshape(-1, t.shape[1])

    mask_flat = mask.reshape(-1).bool()

    if mask_flat.sum() == 0:
        return torch.tensor(0.0, device=student_logits.device)

    loss_per_frame = F.kl_div(log_p_s, p_t, reduction='none').sum(dim=1)
    return loss_per_frame[mask_flat].mean() * (temperature ** 2)


@torch.no_grad()
def evaluate_frame_accuracy(model, loader, model_kind):
    model.eval()
    total_correct = 0
    total_frames = 0

    for batch in loader:
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        text_context = batch['text_context'].to(device)

        if model_kind == 'teacher':
            outputs = model(features, mask, text_context)
        else:
            outputs = model(features, mask)

        preds = torch.argmax(outputs[-1], dim=1)
        valid = mask.bool()

        total_correct += int((preds[valid] == labels[valid]).sum().item())
        total_frames += int(valid.sum().item())

    return 100.0 * total_correct / max(total_frames, 1)


def train_ce_model(model, loader, epochs, model_kind, name):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []

    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        pbar = tqdm(loader, desc=f'{name} epoch {epoch}/{epochs}')
        for batch in pbar:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)
            text_context = batch['text_context'].to(device)

            optimizer.zero_grad()

            if model_kind == 'teacher':
                outputs = model(features, mask, text_context)
            else:
                outputs = model(features, mask)

            loss = supervised_loss(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item())
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        train_acc = evaluate_frame_accuracy(model, train_loader, model_kind)
        test_acc = evaluate_frame_accuracy(model, test_loader, model_kind)

        row = {
            'model': name,
            'epoch': epoch,
            'loss': total_loss / max(len(loader), 1),
            'train_acc': train_acc,
            'test_acc': test_acc,
        }
        history.append(row)
        print(row)

    return pd.DataFrame(history)


def train_kd_student(student, teacher, loader, epochs, name):
    optimizer = torch.optim.Adam(student.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []

    student.to(device)
    teacher.to(device)
    teacher.eval()

    for epoch in range(1, epochs + 1):
        student.train()
        total_loss = 0.0
        total_ce = 0.0
        total_kd = 0.0

        pbar = tqdm(loader, desc=f'{name} epoch {epoch}/{epochs}')
        for batch in pbar:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)
            text_context = batch['text_context'].to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                teacher_outputs = teacher(features, mask, text_context)

            student_outputs = student(features, mask)

            ce = supervised_loss(student_outputs, labels)
            kd = kd_loss(
                student_outputs[-1],
                teacher_outputs[-1].detach(),
                mask,
                temperature=KD_TEMPERATURE,
            )

            loss = LAMBDA_CE * ce + LAMBDA_KD * kd
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item())
            total_ce += float(ce.item())
            total_kd += float(kd.item())

            pbar.set_postfix(loss=f'{loss.item():.4f}', ce=f'{ce.item():.4f}', kd=f'{kd.item():.4f}')

        train_acc = evaluate_frame_accuracy(student, train_loader, 'student')
        test_acc = evaluate_frame_accuracy(student, test_loader, 'student')

        row = {
            'model': name,
            'epoch': epoch,
            'loss': total_loss / max(len(loader), 1),
            'ce': total_ce / max(len(loader), 1),
            'kd': total_kd / max(len(loader), 1),
            'train_acc': train_acc,
            'test_acc': test_acc,
        }
        history.append(row)
        print(row)

    return pd.DataFrame(history)

## 8. Build models

Models:

- `student_ce_only_strong`: stronger video-only CE student.
- `concat_text_teacher`: teacher using visual features plus video-level text context.
- `student_kd_from_concat_teacher`: video-only student distilled from the concat text teacher.

In [ ]:
student_ce_only_strong = MultiStageTCN(
    in_dim=feature_dim,
    num_f_maps=NUM_F_MAPS,
    num_classes=num_classes,
    num_layers=NUM_LAYERS,
    num_stages=NUM_STAGES,
)

concat_text_teacher = ConcatTextTeacher(
    visual_dim=feature_dim,
    text_dim=text_dim,
    num_f_maps=NUM_F_MAPS,
    num_classes=num_classes,
    num_layers=NUM_LAYERS,
    num_stages=NUM_STAGES,
)

student_kd_from_concat_teacher = MultiStageTCN(
    in_dim=feature_dim,
    num_f_maps=NUM_F_MAPS,
    num_classes=num_classes,
    num_layers=NUM_LAYERS,
    num_stages=NUM_STAGES,
)

print(student_ce_only_strong.__class__.__name__)
print(concat_text_teacher.__class__.__name__)
print(student_kd_from_concat_teacher.__class__.__name__)

MultiStageTCN
ConcatTextTeacher
MultiStageTCN


## 9. Train models

For `scale1_control`, this should be relatively quick.

For `full_split1`, this can take much longer because it trains 3 models for 30 epochs each.

In [ ]:
start_time = time.time()

history_ce = train_ce_model(
    student_ce_only_strong,
    train_loader,
    epochs=EPOCHS_CE,
    model_kind='student',
    name='student_ce_only_strong',
)

history_teacher = train_ce_model(
    concat_text_teacher,
    train_loader,
    epochs=EPOCHS_TEACHER,
    model_kind='teacher',
    name='concat_text_teacher',
)


student_kd_from_concat_teacher.load_state_dict(student_ce_only_strong.state_dict())
print("Initialized KD student from CE-only student.")

history_kd = train_kd_student(
    student_kd_from_concat_teacher,
    concat_text_teacher,
    train_loader,
    epochs=EPOCHS_KD,
    name='student_kd_from_concat_teacher',
)

elapsed_min = (time.time() - start_time) / 60.0
print(f'Training completed in {elapsed_min:.2f} minutes')

student_ce_only_strong epoch 1/10:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'student_ce_only_strong', 'epoch': 1, 'loss': 2.5429265842045825, 'train_acc': 29.083153107282925, 'test_acc': 30.922674517531885}


student_ce_only_strong epoch 2/10:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'student_ce_only_strong', 'epoch': 2, 'loss': 1.808954106720343, 'train_acc': 40.08375366272379, 'test_acc': 37.48985995860885}


student_ce_only_strong epoch 3/10:   0%|          | 0/1460 [00:00<?, ?it/s]

...

{'model': 'student_kd_from_concat_teacher', 'epoch': 8, 'loss': 0.5829081400463434, 'ce': 0.5040144514088353, 'kd': 0.7889368857831172, 'train_acc': 86.89651551445692, 'test_acc': 68.36326871406469}


student_kd_from_concat_teacher epoch 9/10:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'student_kd_from_concat_teacher', 'epoch': 9, 'loss': 0.4595703336065763, 'ce': 0.4000495419008275, 'kd': 0.5952079098926831, 'train_acc': 87.82635553595117, 'test_acc': 69.6429914012449}


student_kd_from_concat_teacher epoch 10/10:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'student_kd_from_concat_teacher', 'epoch': 10, 'loss': 0.49590888119315446, 'ce': 0.4298823522786572, 'kd': 0.6602652871139245, 'train_acc': 87.91402431455494, 'test_acc': 64.40716866301823}
Training completed in 173.35 minutes


## 10. Compare frame-wise accuracy

In [ ]:
comparison = pd.DataFrame([
    {
        'model': 'student_ce_only_strong',
        'train_acc': evaluate_frame_accuracy(student_ce_only_strong, train_loader, 'student'),
        'test_acc': evaluate_frame_accuracy(student_ce_only_strong, test_loader, 'student'),
        'text_during_training': 'no',
        'text_during_inference': 'no',
    },
    {
        'model': 'concat_text_teacher',
        'train_acc': evaluate_frame_accuracy(concat_text_teacher, train_loader, 'teacher'),
        'test_acc': evaluate_frame_accuracy(concat_text_teacher, test_loader, 'teacher'),
        'text_during_training': 'yes',
        'text_during_inference': 'yes',
    },
    {
        'model': 'student_kd_from_concat_teacher',
        'train_acc': evaluate_frame_accuracy(student_kd_from_concat_teacher, train_loader, 'student'),
        'test_acc': evaluate_frame_accuracy(student_kd_from_concat_teacher, test_loader, 'student'),
        'text_during_training': 'yes',
        'text_during_inference': 'no',
    },
])

display(comparison)

comparison_path = RUN_ROOT / 'comparison_frame_accuracy.csv'
comparison.to_csv(comparison_path, index=False)
print('Saved:', comparison_path)

,model,train_acc,test_acc,text_during_training,text_during_inference
0,student_ce_only_strong,80.132505,67.295448,no,no
1,concat_text_teacher,85.453594,73.544879,yes,yes
2,student_kd_from_concat_teacher,87.914024,64.407169,yes,no


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/comparison_frame_accuracy.csv


## 11. Save checkpoint and training histories

In [ ]:
checkpoint = {
    'config': {
        'run_mode': RUN_MODE,
        'run_name': RUN_NAME,
        'split_id': SPLIT_ID,
        'num_stages': NUM_STAGES,
        'num_layers': NUM_LAYERS,
        'num_f_maps': NUM_F_MAPS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'lambda_ce': LAMBDA_CE,
        'lambda_kd': LAMBDA_KD,
        'kd_temperature': KD_TEMPERATURE,
    },
    'student_ce_only_strong_state_dict': student_ce_only_strong.state_dict(),
    'concat_text_teacher_state_dict': concat_text_teacher.state_dict(),
    'student_kd_from_concat_teacher_state_dict': student_kd_from_concat_teacher.state_dict(),
    'comparison': comparison.to_dict(orient='records'),
}

ckpt_path = RUN_ROOT / 'improved_concat_teacher_checkpoint.pt'
torch.save(checkpoint, ckpt_path, _use_new_zipfile_serialization=False)
print('Saved checkpoint:', ckpt_path)

history_ce.to_csv(RUN_ROOT / 'history_student_ce_only_strong.csv', index=False)
history_teacher.to_csv(RUN_ROOT / 'history_concat_text_teacher.csv', index=False)
history_kd.to_csv(RUN_ROOT / 'history_student_kd_from_concat_teacher.csv', index=False)

summary = {
    'status': 'completed',
    'run_name': RUN_NAME,
    'run_mode': RUN_MODE,
    'split_id': SPLIT_ID,
    'train_videos': len(train_ids),
    'test_videos': len(test_ids),
    'num_classes': num_classes,
    'feature_dim': feature_dim,
    'text_dim': text_dim,
    'comparison': comparison.to_dict(orient='records'),
    'checkpoint': str(ckpt_path),
}

summary_path = RUN_ROOT / 'experiment_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary:', summary_path)

Saved checkpoint: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/improved_concat_teacher_checkpoint.pt
Saved summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/experiment_summary.json


## 12. Save frame-level predictions

Prediction files are needed for Edit and F1 metrics.

Each file contains one predicted action label per frame.

In [ ]:
@torch.no_grad()
def save_predictions(model, loader, model_name, model_kind):
    model.eval()

    pred_dir = RUN_ROOT / 'predictions' / model_name
    pred_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for batch in tqdm(loader, desc=f'Saving predictions: {model_name}'):
        features = batch['features'].to(device)
        mask = batch['mask'].to(device)
        text_context = batch['text_context'].to(device)
        lengths = batch['lengths'].cpu().numpy()
        video_ids = batch['video_ids']

        if model_kind == 'teacher':
            outputs = model(features, mask, text_context)
        else:
            outputs = model(features, mask)

        preds = torch.argmax(outputs[-1], dim=1).cpu().numpy()

        for i, video_id in enumerate(video_ids):
            T = int(lengths[i])
            pred_idx = preds[i, :T]
            pred_labels = [id_to_label[int(x)] for x in pred_idx]

            out_path = pred_dir / f'{video_id}.txt'
            out_path.write_text('\n'.join(pred_labels) + '\n')

            rows.append({
                'model': model_name,
                'video_id': video_id,
                'num_frames': T,
                'prediction_path': str(out_path),
            })

    manifest = pd.DataFrame(rows)
    manifest_path = RUN_ROOT / f'{model_name}_prediction_manifest.csv'
    manifest.to_csv(manifest_path, index=False)

    print(f'Saved {len(manifest)} predictions for {model_name}')
    print('Manifest:', manifest_path)

    return manifest


prediction_manifests = []

prediction_manifests.append(save_predictions(
    student_ce_only_strong,
    test_loader,
    'student_ce_only_strong',
    'student',
))

prediction_manifests.append(save_predictions(
    concat_text_teacher,
    test_loader,
    'concat_text_teacher',
    'teacher',
))

prediction_manifests.append(save_predictions(
    student_kd_from_concat_teacher,
    test_loader,
    'student_kd_from_concat_teacher',
    'student',
))

df_prediction_manifest = pd.concat(prediction_manifests, ignore_index=True)
all_manifest_path = RUN_ROOT / 'prediction_manifest_all_models.csv'
df_prediction_manifest.to_csv(all_manifest_path, index=False)

display(df_prediction_manifest.head())
print('Saved combined manifest:', all_manifest_path)

Saving predictions: student_ce_only_strong:   0%|          | 0/252 [00:00<?, ?it/s]

Saved 252 predictions for student_ce_only_strong
Manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/student_ce_only_strong_prediction_manifest.csv


Saving predictions: concat_text_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

Saved 252 predictions for concat_text_teacher
Manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/concat_text_teacher_prediction_manifest.csv


Saving predictions: student_kd_from_concat_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

Saved 252 predictions for student_kd_from_concat_teacher
Manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/student_kd_from_concat_teacher_prediction_manifest.csv


,model,video_id,num_frames,prediction_path
0,student_ce_only_strong,P03_cam01_P03_cereals,832,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
1,student_ce_only_strong,P03_cam01_P03_coffee,917,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
2,student_ce_only_strong,P03_cam01_P03_friedegg,4266,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
3,student_ce_only_strong,P03_cam01_P03_milk,1158,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
4,student_ce_only_strong,P03_cam01_P03_salat,4449,/content/drive/MyDrive/mmf_tas_lab_data/text_a...


Saved combined manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/prediction_manifest_all_models.csv


## 13. TAS metric functions

In [ ]:
BACKGROUND_LABELS = {'background', 'SIL', 'sil'}

def read_label_file(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]


def get_segments(frame_labels, background_labels=BACKGROUND_LABELS):
    labels = []
    starts = []
    ends = []

    last_label = None
    start = 0

    for i, label in enumerate(frame_labels):
        if last_label is None:
            last_label = label
            start = i
        elif label != last_label:
            if last_label not in background_labels:
                labels.append(last_label)
                starts.append(start)
                ends.append(i)
            last_label = label
            start = i

    if last_label is not None and last_label not in background_labels:
        labels.append(last_label)
        starts.append(start)
        ends.append(len(frame_labels))

    return labels, np.asarray(starts), np.asarray(ends)


def levenshtein_distance(pred_labels, gt_labels):
    m = len(pred_labels)
    n = len(gt_labels)

    dp = np.zeros((m + 1, n + 1), dtype=np.float32)
    dp[:, 0] = np.arange(m + 1)
    dp[0, :] = np.arange(n + 1)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if pred_labels[i - 1] == gt_labels[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,
                dp[i, j - 1] + 1,
                dp[i - 1, j - 1] + cost,
            )

    return dp[m, n]


def edit_score(pred_frame_labels, gt_frame_labels):
    pred_labels, _, _ = get_segments(pred_frame_labels)
    gt_labels, _, _ = get_segments(gt_frame_labels)

    if len(gt_labels) == 0 and len(pred_labels) == 0:
        return 100.0

    denom = max(len(pred_labels), len(gt_labels))
    if denom == 0:
        return 0.0

    edit = levenshtein_distance(pred_labels, gt_labels)
    return (1.0 - edit / denom) * 100.0


def f_score_video(pred_frame_labels, gt_frame_labels, overlap):
    pred_labels, pred_starts, pred_ends = get_segments(pred_frame_labels)
    gt_labels, gt_starts, gt_ends = get_segments(gt_frame_labels)

    n_pred = len(pred_labels)
    n_gt = len(gt_labels)

    if n_pred == 0 and n_gt == 0:
        return 0, 0, 0
    if n_pred == 0:
        return 0, 0, n_gt
    if n_gt == 0:
        return 0, n_pred, 0

    hits = np.zeros(n_gt, dtype=np.float32)
    tp = 0
    fp = 0

    for j in range(n_pred):
        intersection = np.minimum(pred_ends[j], gt_ends) - np.maximum(pred_starts[j], gt_starts)
        union = np.maximum(pred_ends[j], gt_ends) - np.minimum(pred_starts[j], gt_starts)

        intersection = np.maximum(intersection, 0)
        iou = intersection / np.maximum(union, 1e-8)

        label_match = np.asarray([pred_labels[j] == gt_label for gt_label in gt_labels], dtype=bool)
        iou = iou * label_match

        idx = int(np.argmax(iou))
        if iou[idx] >= overlap and hits[idx] == 0:
            tp += 1
            hits[idx] = 1
        else:
            fp += 1

    fn = n_gt - int(hits.sum())
    return tp, fp, fn

## 14. Evaluate TAS metrics

In [ ]:
def evaluate_tas_model(model_name):
    pred_dir = RUN_ROOT / 'predictions' / model_name
    pred_files = sorted(pred_dir.glob('*.txt'))

    rows = []
    total_correct = 0
    total_frames = 0
    edit_scores = []

    f_counts = {
        0.10: {'tp': 0, 'fp': 0, 'fn': 0},
        0.25: {'tp': 0, 'fp': 0, 'fn': 0},
        0.50: {'tp': 0, 'fp': 0, 'fn': 0},
    }

    for pred_path in tqdm(pred_files, desc=f'Evaluating {model_name}'):
        video_id = pred_path.stem
        gt_path = GT_DIR / f'{video_id}.txt'

        pred_labels = read_label_file(pred_path)
        gt_labels = read_label_file(gt_path)

        n = min(len(pred_labels), len(gt_labels))
        correct = int(np.sum(np.asarray(pred_labels[:n]) == np.asarray(gt_labels[:n])))

        total_correct += correct
        total_frames += n

        edit = edit_score(pred_labels, gt_labels)
        edit_scores.append(edit)

        for overlap in f_counts.keys():
            tp, fp, fn = f_score_video(pred_labels, gt_labels, overlap=overlap)
            f_counts[overlap]['tp'] += tp
            f_counts[overlap]['fp'] += fp
            f_counts[overlap]['fn'] += fn

        rows.append({
            'model': model_name,
            'video_id': video_id,
            'num_pred_frames': len(pred_labels),
            'num_gt_frames': len(gt_labels),
            'num_eval_frames': n,
            'length_difference': len(pred_labels) - len(gt_labels),
            'frame_accuracy': 100.0 * correct / max(n, 1),
            'edit': edit,
        })

    summary = {
        'model': model_name,
        'num_prediction_files': len(pred_files),
        'total_eval_frames': int(total_frames),
        'accuracy': 100.0 * total_correct / max(total_frames, 1),
        'edit': float(np.mean(edit_scores)) if edit_scores else 0.0,
    }

    for overlap, counts in f_counts.items():
        tp = counts['tp']
        fp = counts['fp']
        fn = counts['fn']

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        suffix = int(overlap * 100)
        summary[f'f1@{suffix}'] = f1 * 100.0
        summary[f'precision@{suffix}'] = precision * 100.0
        summary[f'recall@{suffix}'] = recall * 100.0

    return summary, pd.DataFrame(rows)


tas_model_names = [
    'student_ce_only_strong',
    'concat_text_teacher',
    'student_kd_from_concat_teacher',
]

summary_rows = []
video_metric_dfs = []

for model_name in tas_model_names:
    summary, video_df = evaluate_tas_model(model_name)
    summary_rows.append(summary)
    video_metric_dfs.append(video_df)

df_tas_summary = pd.DataFrame(summary_rows)
df_tas_per_video = pd.concat(video_metric_dfs, ignore_index=True)

eval_dir = RUN_ROOT / 'evaluation_metrics'
eval_dir.mkdir(parents=True, exist_ok=True)

tas_summary_path = eval_dir / 'tas_metrics_summary.csv'
tas_per_video_path = eval_dir / 'tas_metrics_per_video.csv'

df_tas_summary.to_csv(tas_summary_path, index=False)
df_tas_per_video.to_csv(tas_per_video_path, index=False)

display(df_tas_summary)

print('Saved:', tas_summary_path)
print('Saved:', tas_per_video_path)

mismatches = df_tas_per_video[df_tas_per_video['length_difference'] != 0]
print('Length mismatches:', len(mismatches))

Evaluating student_ce_only_strong:   0%|          | 0/252 [00:00<?, ?it/s]

Evaluating concat_text_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

Evaluating student_kd_from_concat_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

,model,num_prediction_files,total_eval_frames,accuracy,edit,f1@10,precision@10,recall@10,f1@25,precision@25,recall@25,f1@50,precision@50,recall@50
0,student_ce_only_strong,252,505422,67.295448,48.664978,34.513105,22.875983,70.249221,31.299024,20.745625,63.707165,24.947389,16.535633,50.778816
1,concat_text_teacher,252,505422,73.544879,63.716728,55.004302,43.531548,74.688474,51.390880,40.671811,69.781931,40.607972,32.137994,55.140187
2,student_kd_from_concat_teacher,252,505422,64.407169,49.473190,29.022506,18.501071,67.289720,26.973463,17.194861,62.538941,21.498153,13.704497,49.844237


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/evaluation_metrics/tas_metrics_summary.csv
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/evaluation_metrics/tas_metrics_per_video.csv
Length mismatches: 0


## 15. Save Markdown report

In [ ]:
report_path = RUN_ROOT / 'evaluation_metrics' / 'tas_metrics_report.md'

display_cols = ['model', 'accuracy', 'edit', 'f1@10', 'f1@25', 'f1@50']
df_report = df_tas_summary[display_cols].copy()

for col in ['accuracy', 'edit', 'f1@10', 'f1@25', 'f1@50']:
    df_report[col] = df_report[col].map(lambda x: f'{x:.2f}')

markdown_table = df_report.to_markdown(index=False)

report = f"""# Improved Concat Text Teacher TAS Metrics Report

Run: `{RUN_NAME}`

Dataset: Breakfast

Split: {SPLIT_ID}

Run mode: `{RUN_MODE}`

## Summary

{markdown_table}

## Main comparison

The key comparison is:

```text
student_ce_only_strong
vs
student_kd_from_concat_teacher
```

The KD student uses text only during training through the teacher and remains video-only at inference.

## Files

- Frame accuracy CSV: `{comparison_path}`
- TAS summary CSV: `{tas_summary_path}`
- TAS per-video CSV: `{tas_per_video_path}`
- Checkpoint: `{ckpt_path}`
"""

report_path.write_text(report)

print('Saved:', report_path)
print(report)

Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/improved_concat_teacher_full_split1_split1/evaluation_metrics/tas_metrics_report.md
# Improved Concat Text Teacher TAS Metrics Report

Run: `improved_concat_teacher_full_split1_split1`

Dataset: Breakfast

Split: 1

Run mode: `full_split1`

## Summary

| model                          |   accuracy |   edit |   f1@10 |   f1@25 |   f1@50 |
|:-------------------------------|-----------:|-------:|--------:|--------:|--------:|
| student_ce_only_strong         |      67.3  |  48.66 |   34.51 |   31.3  |   24.95 |
| concat_text_teacher            |      73.54 |  63.72 |   55    |   51.39 |   40.61 |
| student_kd_from_concat_teacher |      64.41 |  49.47 |   29.02 |   26.97 |   21.5  |

## Main comparison

The key comparison is:

```text
student_ce_only_strong
vs
student_kd_from_concat_teacher
```

The KD student uses text only during training through the teacher and remains video-only at inference.

## Files

- F